In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(mid_channels, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Calculate needed padding to match x2 dimensions
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.bilinear = bilinear

        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1   = Up(1024, 512 // factor, bilinear)
        self.up2   = Up(512, 256 // factor, bilinear)
        self.up3   = Up(256, 128 // factor, bilinear)
        self.up4   = Up(128, 64, bilinear)

        # Final fully connected (linear) layer: for each pixel,
        # the 64-dimensional feature vector is mapped to a single value (LST).
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        # Downsampling path
        x1 = self.inc(x)     # (batch, 64, H, W)
        x2 = self.down1(x1)  # (batch, 128, H/2, W/2)
        x3 = self.down2(x2)  # (batch, 256, H/4, W/4)
        x4 = self.down3(x3)  # (batch, 512, H/8, W/8)
        x5 = self.down4(x4)  # (batch, 1024//factor, H/16, W/16)

        # Upsampling path
        x = self.up1(x5, x4)  # (batch, 512//factor, H/8, W/8)
        x = self.up2(x, x3)   # (batch, 256//factor, H/4, W/4)
        x = self.up3(x, x2)   # (batch, 128//factor, H/2, W/2)
        x = self.up4(x, x1)   # (batch, 64, H, W)

        # x has shape (batch, 64, H, W). We now want to apply a linear layer per pixel.
        b, c, h, w = x.shape  # here c is 64
        # Rearrange tensor so that each pixel is a separate sample:
        # New shape: (batch, H, W, 64)
        x = x.permute(0, 2, 3, 1).contiguous()
        # Flatten spatial dimensions: (batch*H*W, 64)
        x = x.view(-1, c)
        # Apply the linear layer (per-pixel regression)
        x = self.fc(x)  # output shape: (batch*H*W, 1)
        # Reshape back to (batch, 1, H, W)
        x = x.view(b, h, w, 1).permute(0, 3, 1, 2)
        return x

    def use_checkpointing(self):
        # Note: Ensure your forward pass supports checkpointing when using these wrappers.
        self.inc   = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1   = torch.utils.checkpoint(self.up1)
        self.up2   = torch.utils.checkpoint(self.up2)
        self.up3   = torch.utils.checkpoint(self.up3)
        self.up4   = torch.utils.checkpoint(self.up4)
        self.fc    = torch.utils.checkpoint(self.fc)

In [ ]:

import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import cv2

class RasterDataset(Dataset):
    def __init__(self, file_list, transform=None, nodata_fill_value=-9999.0):
        """
        Args:
            file_list (list): List of dictionaries. Each dictionary should contain keys:
                'Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif', 'LST.tif'
                with the corresponding file paths to the TIFF files.
            transform (callable, optional): Optional transform to be applied on a sample.
            nodata_fill_value (float, optional): The nodata value expected in the rasters.
        """
        self.file_list = file_list
        self.transform = transform
        self.nodata_fill_value = nodata_fill_value

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # Get the file paths for the sample.
        sample_files = self.file_list[idx]

        # Process the input channels.
        channels = []
        for key in ['Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif']:
            with rasterio.open(sample_files[key]) as src:
                # Read the first band (assuming single-band TIFFs)
                channel = src.read(1).astype(np.float32)
            # For inputs, if there are nodata values (NaNs), fill with 0 (or another imputed value)
            channel = np.where(np.isnan(channel), 0.0, channel)
            channels.append(channel)

        # --- Dynamic resizing within the sample ---
        # Use the shape of the first channel as the reference.
        ref_shape = channels[0].shape  # (H, W)
        fixed_channels = []
        for ch in channels:
            if ch.shape != ref_shape:
                ch_resized = cv2.resize(ch, (ref_shape[1], ref_shape[0]), interpolation=cv2.INTER_LINEAR)
                fixed_channels.append(ch_resized)
            else:
                fixed_channels.append(ch)
        channels = fixed_channels
        # -----------------------------------------
        x = np.stack(channels, axis=0)  # shape: (5, H, W)

        # Process the target (LST) and also create a valid mask.
        with rasterio.open(sample_files['LST.tif']) as src:
            y = src.read(1).astype(np.float32)
        # Create a mask: True for valid pixels, False where y is NaN or equals the nodata value.
        valid_mask = ~np.isnan(y)
        valid_mask = valid_mask & (y != self.nodata_fill_value)
        # For training, fill nodata pixels with a neutral value (e.g., 0) in the target.
        y = np.where(valid_mask, y, 0.0)

        # If the target shape doesn't match the reference, resize both y and valid_mask.
        if y.shape != ref_shape:
            y = cv2.resize(y, (ref_shape[1], ref_shape[0]), interpolation=cv2.INTER_LINEAR)
            # For masks, use nearest-neighbor interpolation.
            valid_mask = cv2.resize(valid_mask.astype(np.uint8), (ref_shape[1], ref_shape[0]), interpolation=cv2.INTER_NEAREST)
            valid_mask = valid_mask.astype(bool)
        # Add a channel dimension to y and valid_mask.
        y = np.expand_dims(y, axis=0)          # shape: (1, H, W)
        valid_mask = np.expand_dims(valid_mask, axis=0)  # shape: (1, H, W)

        # Prepare the sample dictionary.
        sample = {'input': x, 'target': y, 'mask': valid_mask}
        if self.transform:
            sample = self.transform(sample)

        # Convert numpy arrays to torch tensors.
        sample['input'] = torch.from_numpy(sample['input'])
        sample['target'] = torch.from_numpy(sample['target'])
        sample['mask'] = torch.from_numpy(sample['mask'])

        # Check that input and target dimensions match.
        if sample['input'].shape[1:] != sample['target'].shape[1:]:
            raise ValueError("Mismatch between input and target spatial dimensions.")
        return sample

from tqdm import tqdm
import os
def list_files_in_folder(folder_path):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if
             os.path.isfile(os.path.join(folder_path, f))]
    return files

def get_file_paths(folder_path):
    file_paths = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            full_path = os.path.abspath(os.path.join(root, file))
            file_paths.append(full_path)
    return file_paths
file_list = []
allAlbedoPixelFiles = []
for filePath in get_file_paths('./Data/X/less5CloudCover'):
    if 'Albedo' in filePath:
        allAlbedoPixelFiles.append(filePath)
for xPath in tqdm(allAlbedoPixelFiles, desc="Packing X,y to dictionary..."):
    fileParts = xPath.split('/')
    fileName, date, city, cloudCategory, dataType = fileParts[-1], fileParts[-2], fileParts[-3], fileParts[-4], fileParts[-5]
    sceneFiles = list_files_in_folder(os.path.dirname(os.path.abspath(xPath)))
    rasterDict = {}
    for rasterPath in sceneFiles:
        rasterName = rasterPath.split('/')[-1]
        rasterDict[rasterName] = rasterPath
        lstPath = xPath.replace('/X/', '/y/').replace('Albedo.tif', 'LST.tif')
        rasterDict['LST.tif'] = lstPath
    file_list.append(rasterDict)

import random
random.shuffle(file_list)
train_ratio = 0.8
train_size = int(len(file_list) * train_ratio)
train_file_list = file_list[:train_size]
test_file_list = file_list[train_size:]

# Create dataset instances for training and testing
train_dataset = RasterDataset(train_file_list)
test_dataset = RasterDataset(test_file_list)

# Create DataLoaders (batch_size=1 to allow dynamic sizes per sample)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.autograd.set_detect_anomaly(True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(n_channels=5, bilinear=False).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs, best_test_loss = 200, 99999999

with open("training.log", "a") as f:
    f.write("\n--- Training Started ---\n")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc="Training"):
        inputs = batch['input'].to(device)   # shape: (1, 5, H, W)
        targets = batch['target'].to(device)   # shape: (1, 1, H, W)
        mask = batch['mask'].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs[mask], targets[mask])
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_dataset)

    # Evaluation phase
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item() * inputs.size(0)
    test_loss /= len(test_dataset)
    checkpointSaved = False
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_test_loss": best_test_loss
        }
        torch.save(checkpoint, f'BestModel.pth')
        checkpointSaved = True
        print(f"New best test loss: {test_loss:.4f} | Model saved as " + f'BestModel.pth\n')
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Test Loss: {test_loss:.4f}")

    with open("training.log", "a") as f:
        f.write(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Test Loss: {test_loss:.4f}\n")
        if checkpointSaved:
            f.write(f"New best test loss: {test_loss:.4f} | Model saved as " + f'BestModel.pth') #Take input of model and transfer geo meta over

In [ ]:
import math

# Evaluation phase
model.eval()
test_mse = 0.0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        inputs = batch['input'].to(device)
        targets = batch['target'].to(device)
        outputs = model(inputs)
        mse = criterion(outputs, targets)  # MSE for this batch
        test_mse += mse.item() * inputs.size(0)
test_mse /= len(test_dataset)
test_rmse = math.sqrt(test_mse)  # Convert MSE to RMSE

print("Accuracy:", str(test_rmse))
# Assume model is already loaded and in evaluation mode.
model.eval()

# Get one sample from the test_loader for inference (batch_size=1).
with torch.no_grad():
    sample = next(iter(test_loader))
    inputs = sample['input'].to(device)  # shape: (1, 5, H, W)
    # Extract the valid mask (assumed to have shape (1, 1, H, W)) and convert it to numpy.
    mask = sample['mask'].to(device)  # valid mask tensor
    mask_np = mask.cpu().numpy().squeeze()  # shape: (H, W)

    # Extract the original LST file path from your test file list.
    lst_tif_path = test_file_list[0]['LST.tif']  # Use first sample's LST path

    # Run inference.
    outputs = model(inputs)  # shape: (1, 1, H, W)
    predicted = outputs.cpu().numpy().squeeze()  # shape: (H, W)

    # Replace positions where the mask is False with np.nan.
    predicted[~mask_np] = np.nan

    # Retrieve the geospatial metadata from the original LST raster.
    with rasterio.open(lst_tif_path) as src:
        profile = src.profile.copy()
        # Optionally, update the nodata value in the metadata.
        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

    # Save the predicted output as a georeferenced TIFF.
    output_filename = "predicted_LST.tif"
    with rasterio.open(output_filename, "w", **profile) as dst:
        dst.write(predicted.astype(np.float32), 1)

print(f"Saved predicted LST raster: {output_filename}")